In [2]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

print("Libraries imported!")
print(f"Pandas Version: {pd.__version__}")
print(f"Numpy Version: {np.__version__}")

Libraries imported!
Pandas Version: 2.2.3
Numpy Version: 2.2.5


In [4]:
data = {
    "policy_id":    ["POL001","POL002","POL003",
                     "POL004","POL005","POL006",
                     "POL007","POL008","POL009","POL010"],
    "holder":       ["Suresh Kumar","Raj Patel",
                     "Priya Singh","Ankit Shah",
                     "Sneha Reddy","Meera Nair",
                     "Vikram Das","Pooja Sharma",
                     "Arjun Menon","Divya Iyer"],
    "policy_type":  ["Health","Auto","Cyber","Health",
                     "Auto","Property","Health","Cyber",
                     "Property","Auto"],
    "premium":      [4500,2300,5600,1800,3200,
                     4700,2100,6200,3800,5100],
    "claim_amount": [15000,8500,22000,5000,12000,
                     9500,45000,31000,7500,18000],
    "status":       ["Active","Inactive","Active",
                     "Active","Renewed","Active",
                     "Inactive","Active","Renewed","Active"],
    "state":        ["Karnataka","Maharashtra",
                     "Tamil Nadu","Gujarat","Telangana",
                     "Kerala","West Bengal","Delhi",
                     "Kerala","Tamil Nadu"],
    "created_date": ["2024-01-15","2024-02-20",
                     "2024-03-10","2024-04-05",
                     "2024-05-12","2024-06-18",
                     "2024-07-22","2024-08-30",
                     "2024-09-14","2024-10-25"]
}

df = pd.DataFrame(data)
df["created_date"] = pd.to_datetime(df["created_date"])
print(f"Data created: {df.shape}")
df.head()

Data created: (10, 8)


,policy_id,holder,policy_type,premium,claim_amount,status,state,created_date
0,POL001,Suresh Kumar,Health,4500,15000,Active,Karnataka,2024-01-15
1,POL002,Raj Patel,Auto,2300,8500,Inactive,Maharashtra,2024-02-20
2,POL003,Priya Singh,Cyber,5600,22000,Active,Tamil Nadu,2024-03-10
3,POL004,Ankit Shah,Health,1800,5000,Active,Gujarat,2024-04-05
4,POL005,Sneha Reddy,Auto,3200,12000,Renewed,Telangana,2024-05-12


In [10]:
print("========== BASIC INFO =============")
print(f"Shape: {df.shape}")
print(f"\nData Types: \n{df.dtypes}")
print(f"\nMissing values: \n{df.isnull().sum()}")
print(f"\nStatus Counts: \n{df["status"].value_counts()}")
print(f"\nPolicy Types: \n{df["policy_type"].value_counts()}")
df.describe().round(2)

========== BASIC INFO =============
Shape: (10, 8)

Data Types: 
policy_id               object
holder                  object
policy_type             object
premium                  int64
claim_amount             int64
status                  object
state                   object
created_date    datetime64[ns]
dtype: object

Missing values: 
policy_id       0
holder          0
policy_type     0
premium         0
claim_amount    0
status          0
state           0
created_date    0
dtype: int64

Status Counts: 
status
Active      6
Inactive    2
Renewed     2
Name: count, dtype: int64

Policy Types: 
policy_type
Health      3
Auto        3
Cyber       2
Property    2
Name: count, dtype: int64


,premium,claim_amount,created_date
count,10.00,10.0,10
mean,3930.00,17350.0,2024-06-01 21:36:00
min,1800.00,5000.0,2024-01-15 00:00:00
25%,2525.00,8750.0,2024-03-16 12:00:00
50%,4150.00,13500.0,2024-05-30 12:00:00
75%,5000.00,21000.0,2024-08-20 06:00:00
max,6200.00,45000.0,2024-10-25 00:00:00
std,1539.16,12456.7,NaN


Filter & Transform

In [13]:
# Filter active policies

active_df = df[
    df["status"].isin(["Active","Renewed"])
].copy().reset_index(drop=True)

print(f"Active/Renewed: {len(active_df)} policies")

# Risk Category

active_df["risk"] = np.where(
    active_df["claim_amount"] > 20000, "High",
    np.where(
        active_df["claim_amount"] >10000,
        "Medium","Low"
    )
)

# Premium Hike

active_df["new_premium"] = np.where(
    active_df["risk"] == "High",
    active_df["premium"]*1.20,
    np.where(
        active_df["risk"] == "Medium",
        active_df["premium"] * 1.10,
        active_df["premium"] * 1.05
    )
).round(2)

# Tax

active_df["tax"] = (
    active_df["new_premium"] * 0.18
).round(2)

# Total Premium
active_df["total_premium"] = (
    active_df["new_premium"]+active_df["tax"]
).round(2)

# Calim ratio

active_df["claim_ratio"] = (
    active_df["claim_amount"]/active_df["premium"]
).round(2)

active_df

Active/Renewed: 8 policies


,policy_id,holder,policy_type,premium,claim_amount,status,state,created_date,risk,new_premium,tax,total_premium,claim_ratio
0,POL001,Suresh Kumar,Health,4500,15000,Active,Karnataka,2024-01-15,Medium,4950.0,891.0,5841.0,3.33
1,POL003,Priya Singh,Cyber,5600,22000,Active,Tamil Nadu,2024-03-10,High,6720.0,1209.6,7929.6,3.93
2,POL004,Ankit Shah,Health,1800,5000,Active,Gujarat,2024-04-05,Low,1890.0,340.2,2230.2,2.78
3,POL005,Sneha Reddy,Auto,3200,12000,Renewed,Telangana,2024-05-12,Medium,3520.0,633.6,4153.6,3.75
4,POL006,Meera Nair,Property,4700,9500,Active,Kerala,2024-06-18,Low,4935.0,888.3,5823.3,2.02
5,POL008,Pooja Sharma,Cyber,6200,31000,Active,Delhi,2024-08-30,High,7440.0,1339.2,8779.2,5.00
6,POL009,Arjun Menon,Property,3800,7500,Renewed,Kerala,2024-09-14,Low,3990.0,718.2,4708.2,1.97
7,POL010,Divya Iyer,Auto,5100,18000,Active,Tamil Nadu,2024-10-25,Medium,5610.0,1009.8,6619.8,3.53


GroupBy Analysis

In [19]:
print("======== POLICY TYPE ANALYSIS ==============")

type_analysis = active_df.groupby("policy_type").agg(
    count = ("policy_id","count"),
    avg_premium = ("premium","mean"),
    total_premium = ("premium","sum"),
    avg_claim = ("claim_amount","mean")
).round(2)
print(type_analysis)

print("\n========== STATE ANALYSIS ================")

state_analysis = active_df.groupby("state").agg(
    count = ("policy_id","count"),
    total_premium = ("premium","sum"),
    avg_claim = ("claim_amount","mean")
).round(2).sort_values("total_premium", ascending=False)
print(state_analysis)


======== POLICY TYPE ANALYSIS ==============
             count  avg_premium  total_premium  avg_claim
policy_type                                              
Auto             2       4150.0           8300    15000.0
Cyber            2       5900.0          11800    26500.0
Health           2       3150.0           6300    10000.0
Property         2       4250.0           8500     8500.0

========== STATE ANALYSIS ================
            count  total_premium  avg_claim
state                                      
Tamil Nadu      2          10700    20000.0
Kerala          2           8500     8500.0
Delhi           1           6200    31000.0
Karnataka       1           4500    15000.0
Telangana       1           3200    12000.0
Gujarat         1           1800     5000.0


Merge Policies & Claims



In [21]:
claims_data = {
    "claim_id":    ["CLM001","CLM002","CLM003",
                    "CLM004","CLM005"],
    "policy_id":   ["POL001","POL003","POL004",
                    "POL006","POL008"],
    "claim_status":["Approved","Rejected","Approved",
                    "Pending","Approved"]
}
claims_df = pd.DataFrame(claims_data)

merged = pd.merge(
    active_df, claims_df,
    on="policy_id",
    how="left"
)
merged

,policy_id,holder,policy_type,premium,claim_amount,status,state,created_date,risk,new_premium,tax,total_premium,claim_ratio,claim_id,claim_status
0,POL001,Suresh Kumar,Health,4500,15000,Active,Karnataka,2024-01-15,Medium,4950.0,891.0,5841.0,3.33,CLM001,Approved
1,POL003,Priya Singh,Cyber,5600,22000,Active,Tamil Nadu,2024-03-10,High,6720.0,1209.6,7929.6,3.93,CLM002,Rejected
2,POL004,Ankit Shah,Health,1800,5000,Active,Gujarat,2024-04-05,Low,1890.0,340.2,2230.2,2.78,CLM003,Approved
3,POL005,Sneha Reddy,Auto,3200,12000,Renewed,Telangana,2024-05-12,Medium,3520.0,633.6,4153.6,3.75,NaN,NaN
4,POL006,Meera Nair,Property,4700,9500,Active,Kerala,2024-06-18,Low,4935.0,888.3,5823.3,2.02,CLM004,Pending
5,POL008,Pooja Sharma,Cyber,6200,31000,Active,Delhi,2024-08-30,High,7440.0,1339.2,8779.2,5.00,CLM005,Approved
6,POL009,Arjun Menon,Property,3800,7500,Renewed,Kerala,2024-09-14,Low,3990.0,718.2,4708.2,1.97,NaN,NaN
7,POL010,Divya Iyer,Auto,5100,18000,Active,Tamil Nadu,2024-10-25,Medium,5610.0,1009.8,6619.8,3.53,NaN,NaN


In [25]:
merged.fillna({
    "claim_id": "NO_CLAIM",
    "claim_status": "NO_CLAIM"
}, inplace=True)

print(f"Merged Shape: {merged.shape}")

merged[["policy_id","holder","claim_id","claim_status","risk"]]

Merged Shape: (8, 15)


,policy_id,holder,claim_id,claim_status,risk
0,POL001,Suresh Kumar,CLM001,Approved,Medium
1,POL003,Priya Singh,CLM002,Rejected,High
2,POL004,Ankit Shah,CLM003,Approved,Low
3,POL005,Sneha Reddy,NO_CLAIM,NO_CLAIM,Medium
4,POL006,Meera Nair,CLM004,Pending,Low
5,POL008,Pooja Sharma,CLM005,Approved,High
6,POL009,Arjun Menon,NO_CLAIM,NO_CLAIM,Low
7,POL010,Divya Iyer,NO_CLAIM,NO_CLAIM,Medium


Save Output

In [26]:
os.makedirs("output", exist_ok=True)

# Save to CSV
active_df.to_csv("output/processed_policies.csv", index=False)

# Save to JSON
active_df.to_json("output/processed_policies.json",
                  orient="records",
                  indent=4
                  )
# Sumamry Report

summary = {
    "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "total_policies": int(len(active_df)),
    "total_premium": float(active_df["premium"].sum()),
    "avg_premium": float(active_df["premium"].mean()),
    "total_claims": float(active_df["claim_amount"].sum()),
    "risk_distribution": {
        "High": int((active_df["risk"]=="High").sum()),
        "Medium": int((active_df["risk"]=="Medium").sum()),
        "Low": int((active_df["risk"]=="Low").sum())
    }
}

with open("output/summary_report.json","w") as f:
    json.dump(summary,f, indent=4)

print("CSV saved!")
print("JSON saved!")
print("Summary saved!")
print("ETL Pipeline Complete")

CSV saved!
JSON saved!
Summary saved!
ETL Pipeline Complete
